# TensorFlow Fundus Image Classification — Training & Weight Export

This notebook trains **6 TensorFlow models** on the fundus image dataset and
exports their weights as `.keras` files for the Streamlit app.

**Models:**
- DenseNet121 (2017)
- EfficientNetB3 (2019)
- EfficientNetV2S (2021)
- ConvNeXtTiny (2022)
- MobileNetV2 (2018)
- ResNet18 (Custom, from scratch)

**Run this on Google Colab with GPU enabled.**

In [ ]:
import os
import json
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import (
    DenseNet121,
    EfficientNetB3,
    EfficientNetV2S,
    ConvNeXtTiny,
    MobileNetV2,
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────

CLASS_NAMES = [
    "AMD", "Bleeding", "Blur Fundus", "Cataract", "Coats", "Cotton Wool Spots",
    "Diabetic Retinopathy", "Drusen", "Glaucoma", "Hemorrhage",
    "Healthy", "Hypertensive Retinopathy", "Laser Scars", "Macular Hole",
    "Maculopathy", "Myopia", "Normal Fundus", "Optic Disc Pallor",
    "Preretinal Hemorrhage", "Proliferative DR", "Retinal Detachment",
    "Retinitis Pigmentosa", "Retinoblastoma", "STARE Normal",
    "Toxoplasmosis", "Vessel Tortuosity",
]

NUM_CLASSES = len(CLASS_NAMES)
IMG_HEIGHT, IMG_WIDTH = 120, 150
BATCH_SIZE = 32
EPOCHS = 20

print(f"Classes: {NUM_CLASSES}")

## Upload / Mount Your Dataset

Update `DATASET_ROOT` to point to your fundus image dataset.
Expected structure:
```
dataset/
  train/
    AMD/
      img001.jpg
      ...
    Bleeding/
      ...
  test/
    AMD/
      ...
```

In [ ]:
# Mount Google Drive (uncomment if your dataset is on Drive)
# from google.colab import drive
# drive.mount('/content/drive')

DATASET_ROOT = "./dataset"  # Update this path
TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
TEST_DIR = os.path.join(DATASET_ROOT, "test")

assert os.path.isdir(TRAIN_DIR), f"Train directory not found: {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR), f"Test directory not found: {TEST_DIR}"

In [ ]:
# ── Data generators ────────────────────────────────────────────────────────

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=15,
    brightness_range=[0.8, 1.2],
    validation_split=0.15,
)

test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASS_NAMES,
    subset="training",
    shuffle=True,
)

val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASS_NAMES,
    subset="validation",
    shuffle=False,
)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASS_NAMES,
    shuffle=False,
)

print(f"Train batches: {len(train_gen)}")
print(f"Val batches:   {len(val_gen)}")
print(f"Test batches:  {len(test_gen)}")

In [ ]:
# ── Custom ResNet18 (from scratch) ────────────────────────────────────────

def _residual_block(x, filters, stride=1):
    shortcut = x
    x = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, strides=1, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding="same", use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x


def build_custom_resnet18(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(64, 7, strides=2, padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)
    for filters, stride in [(64, 1), (64, 1)]:
        x = _residual_block(x, 64, stride)
    for i, stride in enumerate([2, 1]):
        x = _residual_block(x, 128, stride)
    for i, stride in enumerate([2, 1]):
        x = _residual_block(x, 256, stride)
    for i, stride in enumerate([2, 1]):
        x = _residual_block(x, 512, stride)
    x = layers.GlobalAveragePooling2D()(x)
    return tf.keras.Model(inputs, x, name="custom_resnet18")


print("ResNet18 builder ready.")

In [ ]:
# ── Model builder ─────────────────────────────────────────────────────────

MODELS = {
    "DenseNet121":       {"builder": DenseNet121,       "custom": False},
    "EfficientNetB3":    {"builder": EfficientNetB3,    "custom": False},
    "EfficientNetV2S":   {"builder": EfficientNetV2S,   "custom": False},
    "ConvNeXtTiny":      {"builder": ConvNeXtTiny,      "custom": False},
    "MobileNetV2":       {"builder": MobileNetV2,       "custom": False},
    "ResNet18 (Custom)": {"builder": None,              "custom": True},
}


def build_model(name, info):
    """Build a classifier with the given backbone."""
    if info["custom"]:
        base = build_custom_resnet18((IMG_HEIGHT, IMG_WIDTH, 3))
        head = []
    else:
        base = info["builder"](
            weights="imagenet",
            include_top=False,
            input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
        )
        # Unfreeze top layers for fine-tuning
        base.trainable = True
        # Freeze early layers, fine-tune later layers
        total_layers = len(base.layers)
        freeze_until = int(total_layers * 0.7)
        for layer in base.layers[:freeze_until]:
            layer.trainable = False
        head = [layers.GlobalAveragePooling2D()]

    model = models.Sequential([
        base,
        *head,
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


print("Model builder ready.")

In [ ]:
# ── Train all models ──────────────────────────────────────────────────────

os.makedirs("weights", exist_ok=True)
all_results = {}

for model_name, info in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")

    model = build_model(model_name, info)
    model.summary()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=5, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6
        ),
    ]

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=callbacks,
    )

    # Evaluate on test set
    print(f"\nEvaluating {model_name} on test set...")
    test_loss, test_acc = model.evaluate(test_gen)

    # Inference timing
    sample_batch = next(iter(test_gen))[0][:1]
    model.predict(sample_batch, verbose=0)  # warm-up
    start = time.perf_counter()
    for _ in range(50):
        model.predict(sample_batch, verbose=0)
    inf_ms = (time.perf_counter() - start) / 50 * 1000

    all_results[model_name] = {
        "test_accuracy": float(test_acc),
        "test_loss": float(test_loss),
        "inference_ms": float(inf_ms),
        "best_val_accuracy": float(max(history.history["val_accuracy"])),
        "epochs_trained": len(history.history["loss"]),
    }

    # Save weights
    safe_name = model_name.replace(" ", "_").replace("(", "").replace(")", "")
    weight_path = f"weights/{safe_name}.keras"
    model.save(weight_path)
    print(f"  Saved: {weight_path}")
    print(f"  Test Accuracy: {test_acc:.4f}")
    print(f"  Inference:     {inf_ms:.1f} ms/image")

    # Free memory
    del model
    tf.keras.backend.clear_session()

print("\nAll models trained!")

In [ ]:
# ── Export results ────────────────────────────────────────────────────────

with open("weights/tf_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("Results saved to weights/tf_results.json")
print(json.dumps(all_results, indent=2))

In [ ]:
# ── Download weights ──────────────────────────────────────────────────────
# Zip the weights folder for easy download

import shutil
shutil.make_archive("tf_weights", "zip", ".", "weights")

try:
    from google.colab import files
    files.download("tf_weights.zip")
    print("Download triggered!")
except ImportError:
    print("Not on Colab — copy the weights/ folder manually.")

print("\nFiles to place in your intelligenteye/ project:")
print("  weights/DenseNet121.keras")
print("  weights/EfficientNetB3.keras")
print("  weights/EfficientNetV2S.keras")
print("  weights/ConvNeXtTiny.keras")
print("  weights/MobileNetV2.keras")
print("  weights/ResNet18_Custom.keras")
print("  weights/tf_results.json")

## Next Steps

1. Download `tf_weights.zip` from the output above
2. Extract the `weights/` folder into your `intelligenteye/` project root
3. Run `streamlit run app.py` — models will now load trained weights and give real predictions